# Arabic Diacritization Benchmark 
### on Glonor-ByT5-Arabic & Flan-T5-Tashkeel-Small Models 

Compares Arabic diacritization (tashkeel) seq2seq models — Flan-T5-based and ByT5-based — on the `Misraj/SadeedDiac-25` benchmark, split into Modern Standard Arabic (MSA) and Classical Arabic (CA) subsets.

## 1. Environment & Imports

In [ ]:
!pip install -q transformers datasets torch accelerate jiwer

import re
import sys
import torch
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
from tqdm import tqdm
import jiwer  # actually used now, see compute_der_wer()

# --- Config ---
SAMPLE_SIZE = 600

# Keep a running list of anything that gets skipped, so the final report
# is explicit about what's missing and why, instead of a silent gap.
skipped_models = []

## 2. Setup ARABIC DIACRITICS

In [2]:
# Regex Pattern to catch explicit Arabic diacritics (harakat)
ARABIC_DIACRITICS = re.compile(r'[\u064B-\u0652]')

## 3. String Normalization & Metrics Core

In [3]:
def strip_diacritics(text: str) -> str:
    """Extracts raw consonantal skeleton text by eliminating vowels."""
    return ARABIC_DIACRITICS.sub('', text)

def clean_and_tokenize(text: str) -> list:
    """Strips alphanumeric noise and splits text block into sequential words."""
    cleaned = re.sub(r'[^\w\s\u064B-\u0652]', '', text)
    return cleaned.split()

def strip_case_ending(word: str) -> str:
    """
    Removes the diacritic(s) sitting on the word-final letter (i'rab / case
    ending), leaving any diacritics earlier in the word untouched.

    Arabic diacritics are combining marks that trail the base letter they
    attach to, so "case ending" = whatever diacritic chars follow the last
    non-diacritic character in the word.
    """
    last_base_idx = None
    for i, ch in enumerate(word):
        if not ARABIC_DIACRITICS.match(ch):
            last_base_idx = i
    if last_base_idx is None:
        return word
    return word[:last_base_idx + 1]

def _align_words(predictions: list, references: list):
    """
    Shared word-alignment step used by both DER and WER below, so a
    dropped/inserted word doesn't cascade into false mismatches for
    everything after it (see compute_der docstring for why this matters).
    Yields (ref_word, pred_word_or_None) pairs: pred is None for deletions.
    """
    for pred, ref in zip(predictions, references):
        pred_words = clean_and_tokenize(pred)
        ref_words = clean_and_tokenize(ref)
        pred_skel = [strip_diacritics(w) for w in pred_words]
        ref_skel = [strip_diacritics(w) for w in ref_words]

        alignment = jiwer.process_words(" ".join(ref_skel), " ".join(pred_skel))

        for chunk in alignment.alignments[0]:
            if chunk.type == "equal":
                for i in range(chunk.ref_start_idx, chunk.ref_end_idx):
                    r_word = ref_words[i]
                    p_word = pred_words[chunk.hyp_start_idx + (i - chunk.ref_start_idx)]
                    yield r_word, p_word
            elif chunk.type in ("substitute", "delete"):
                # Skeleton didn't match (or word is missing) -> every
                # diacritic slot in the reference word counts as wrong.
                for i in range(chunk.ref_start_idx, chunk.ref_end_idx):
                    yield ref_words[i], None
            # "insert" chunks (hallucinated extra words) have no reference
            # word to score diacritics against, so they're skipped here.

def compute_der(predictions: list, references: list, ce: bool = True) -> float:
    """
    Diacritic Error Rate computed with proper edit-distance word alignment
    (via jiwer's alignment output) instead of naive positional indexing.

    ce=True  -> standard DER, scores every diacritic including the case
                ending (the mark on each word's final letter).
    ce=False -> DER*, the case-ending diacritic is stripped from both sides
                before comparing, so errors that are *only* about i'rab
                don't count. This isolates "core" diacritization quality
                from the notoriously hard, often syntax-dependent case
                ending, matching how DER/DER* is usually reported in the
                Arabic diacritization literature.
    """
    total_chars, wrong_chars = 0, 0

    for r_word, p_word in _align_words(predictions, references):
        if ce is False:
            r_word = strip_case_ending(r_word)
            if p_word is not None:
                p_word = strip_case_ending(p_word)

        r_diacs = ARABIC_DIACRITICS.findall(r_word)
        n = max(len(r_diacs), 1)

        if p_word is None:
            # Deleted / unmatched word: every reference diacritic slot is wrong.
            total_chars += n
            wrong_chars += n
            continue

        p_diacs = ARABIC_DIACRITICS.findall(p_word)
        n = max(len(r_diacs), len(p_diacs), 1)
        total_chars += n
        if r_diacs != p_diacs:
            mism = sum(1 for a, b in zip(r_diacs, p_diacs) if a != b)
            mism += abs(len(r_diacs) - len(p_diacs))
            wrong_chars += max(mism, 1)

    return round((wrong_chars / total_chars) * 100, 2) if total_chars > 0 else 0.0

def compute_wer(predictions: list, references: list, ce: bool = True) -> float:
    """
    Word-level diacritization error rate: a word counts as correct only if
    its full diacritic pattern matches the reference exactly (not just the
    underlying letters). This is the standard "WER" reported alongside DER
    in diacritization papers.

    ce=True  -> a mismatch anywhere in the word (including the case ending)
                marks the word wrong.
    ce=False -> WER*, the case-ending diacritic is stripped from both sides
                first, so a word that's only "wrong" on its i'rab is not
                counted as an error.
    """
    total_words, wrong_words = 0, 0

    for r_word, p_word in _align_words(predictions, references):
        total_words += 1
        if p_word is None:
            wrong_words += 1
            continue
        if ce is False:
            r_word = strip_case_ending(r_word)
            p_word = strip_case_ending(p_word)
        if r_word != p_word:
            wrong_words += 1

    return round((wrong_words / total_words) * 100, 2) if total_words > 0 else 0.0

def compute_der_wer(predictions: list, references: list) -> dict:
    """
    Returns all four headline metrics at once:
    DER / WER with case ending (standard) and DER* / WER* without it.
    """
    return {
        "DER_ce": compute_der(predictions, references, ce=True),
        "DER_noce": compute_der(predictions, references, ce=False),
        "WER_ce": compute_wer(predictions, references, ce=True),
        "WER_noce": compute_wer(predictions, references, ce=False),
    }


## 4. Dataset Parsing (MSA vs. Classical Arabic)

In [ ]:
print("Downloading and processing SadeedDiac-25 Dataset...")
dataset = load_dataset("Misraj/SadeedDiac-25", split="train")

msa_samples, ca_samples = [], []

for item in dataset:
    text_content = item.get("input", "")
    texy_Output = item.get("output","")
    domain = item.get("filename", "").lower()
    source = item.get("filename", "").lower()

    sample = {"ground_truth": texy_Output, "raw_input": strip_diacritics(text_content)}

    if "fadel" in source or domain in ["religion", "classical_poetry", "hadith"]:
        ca_samples.append(sample)
    else:
        msa_samples.append(sample)

msa_test_set = msa_samples[:SAMPLE_SIZE]
ca_test_set = ca_samples[:SAMPLE_SIZE]
print(f"Batches initialized. Using {SAMPLE_SIZE} samples per domain "
      f"(MSA available: {len(msa_samples)}, CA available: {len(ca_samples)}).")
if SAMPLE_SIZE < 200:
    print("SAMPLE_SIZE is small — treat DER/WER below as a smoke test, "
          "not a citable benchmark result.")

Batches initialized. Using 600 samples per domain (MSA available: 600, CA available: 600).


## 5. Pipeline Execution Wrappers

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

def run_hf_seq2seq(model_id, test_dataset, desc_tag):
    """Inference wrapper for ByT5 / Flan-T5 text2text diacritization models."""
    print(f"Loading {model_id} onto target hardware device...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)
        model.eval()
    except Exception as e:
        print(f"\u274c Pipeline skipped for model {model_id}: {e}")
        skipped_models.append((desc_tag, str(e)))
        return None

    predictions = []
    references = [x["ground_truth"] for x in test_dataset]

    for item in tqdm(test_dataset, desc=f"Running {desc_tag}"):
        inputs = tokenizer(item["raw_input"], return_tensors="pt",
                            max_length=1024, truncation=True).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=1024)
        predictions.append(tokenizer.decode(outputs[0], skip_special_tokens=True))

    return compute_der_wer(predictions, references)


## 6. Core Benchmarking Control Loop

In [ ]:
model_registry = {
    "Flan-T5-Tashkeel-Small": {"repo": "Abdou/arabic-tashkeel-flan-t5-small", "type": "seq2seq"},
    "Glonor-ByT5-Arabic": {"repo": "glonor/byt5-arabic-diacritization", "type": "seq2seq"}
}

benchmark_records = []

for domain, test_set in [("MSA (Modern)", msa_test_set), ("CA (Classical)", ca_test_set)]:
    if not test_set:
        print(f"\u26a0\ufe0f  No samples found for {domain} \u2014 skipping this domain entirely.")
        continue

    print(f"\n\U0001f680 ACTIVATING PARALLEL PERFORMANCE EVALUATION LOOP FOR: {domain}")

    for label, meta in model_registry.items():
        metrics = run_hf_seq2seq(meta["repo"], test_set, label)

        if metrics is not None:
            benchmark_records.append({
                "Model Structural ID": label,
                "Domain Track": domain,
                "DER (%) - With Case Ending": metrics["DER_ce"],
                "DER (%) - Without Case Ending": metrics["DER_noce"],
                "WER (%) - With Case Ending": metrics["WER_ce"],
                "WER (%) - Without Case Ending": metrics["WER_noce"],
            })



🚀 ACTIVATING PARALLEL PERFORMANCE EVALUATION LOOP FOR: MSA (Modern)
Loading Abdou/arabic-tashkeel-flan-t5-small onto target hardware device...


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Running Flan-T5-Tashkeel-Small: 100%|██████████| 600/600 [32:41<00:00,  3.27s/it]


Loading glonor/byt5-arabic-diacritization onto target hardware device...


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Running Flan-T5-Tashkeel-Small: 100%|██████████| 600/600 [21:39<00:00,  2.17s/it]


Loading glonor/byt5-arabic-diacritization onto target hardware device...


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
Running Glonor-ByT5-Arabic: 100%|██████████| 600/600 [3:33:34<00:00, 21.36s/it]


## 7. Generate Performance Report Table

In [12]:
performance_df = pd.DataFrame(benchmark_records)
print("\n================ COMPREHENSIVE BENCHMARKING RECAP ================")

if performance_df.empty:
    print("No models produced results \u2014 check the skip log below.")
else:
    print(performance_df.to_markdown(index=False))

    # --- Mean row per model: average of MSA and CA (i.e. across domains) ---
    metric_cols = [
        "DER (%) - With Case Ending",
        "DER (%) - Without Case Ending",
        "WER (%) - With Case Ending",
        "WER (%) - Without Case Ending",
    ]
    mean_df = (
        performance_df
        .groupby("Model Structural ID")[metric_cols]
        .mean()
        .round(2)
        .reset_index()
    )
    mean_df.insert(1, "Domain Track", "Mean (MSA + CA)")

    full_df = pd.concat([performance_df, mean_df], ignore_index=True)
    full_df["Domain Track"] = pd.Categorical(
        full_df["Domain Track"],
        categories=["MSA (Modern)", "CA (Classical)", "Mean (MSA + CA)"],
        ordered=True,
    )
    full_df = full_df.sort_values(["Model Structural ID", "Domain Track"]).reset_index(drop=True)

    print("\n================ WITH MSA / CA MEAN ROW ================")
    print(full_df.to_markdown(index=False))

if skipped_models:
    print("\n================ SKIPPED / FAILED MODELS (explicit) ================")
    for name, reason in skipped_models:
        print(f"- {name}: {reason}")
else:
    print("\nAll registered models ran successfully \u2014 no gaps in the table above.")



================ COMPREHENSIVE BENCHMARKING RECAP ================
| Model Structural ID    | Domain Track   |   DER (%) - With Case Ending |   DER (%) - Without Case Ending |   WER (%) - With Case Ending |   WER (%) - Without Case Ending |
|:-----------------------|:---------------|-----------------------------:|--------------------------------:|-----------------------------:|--------------------------------:|
| Flan-T5-Tashkeel-Small | MSA (Modern)   |                        11.14 |                            9.96 |                        20.42 |                           14.06 |
| Glonor-ByT5-Arabic     | MSA (Modern)   |                        40.04 |                           37.61 |                        49.86 |                           41.97 |
| Flan-T5-Tashkeel-Small | CA (Classical) |                        11.9  |                           11.16 |                        19.45 |                           14.19 |
| Glonor-ByT5-Arabic     | CA (Classical) |                   